In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
df_raw = pd.read_csv("../data/raw/ott_movies_data.csv")
print("Raw dataset shape:", df_raw.shape)
df_raw.head()

ImportError: DLL load failed while importing _ujson: An Application Control policy has blocked this file.

In [ ]:
df_raw.info()
print("\nMissing values per column:\n", df_raw.isnull().sum())
print("\nExact duplicate rows:", df_raw.duplicated().sum())
df_raw.describe()

NameError: name 'df_raw' is not defined

In [ ]:
df = df_raw.drop_duplicates().copy()
for col in ["City", "Subscription_Plan", "Device_Type", "Genre"]:
    df[col] = df[col].astype(str).str.strip()

df["Watch_Date"] = pd.to_datetime(df["Watch_Date"])
print("Cleaned dataset shape:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())

In [ ]:
df["Spend_Per_Minute_INR"] = (df["Monthly_Spend_INR"] / df["Watch_Duration_Mins"]).round(2)


def classify_retention(score):
    if score < 50.0:
        return "At Risk"
    elif score < 75.0:
        return "Moderate"
    return "High Engagement"


df["Retention_Status"] = df["Engagement_Score"].apply(classify_retention)
df["Duration_Bucket"] = pd.cut(
    df["Watch_Duration_Mins"],
    bins=[0, 60, 110, 240],
    labels=["Short (<=60m)", "Medium (61-110m)", "Long (>110m)"],
)
df[
    [
        "Session_ID",
        "Watch_Duration_Mins",
        "Monthly_Spend_INR",
        "Spend_Per_Minute_INR",
        "Retention_Status",
        "Duration_Bucket",
    ]
].head()

In [ ]:
print(df["Engagement_Score"].describe().round(2))

plt.figure(figsize=(9, 5))
sns.histplot(df["Engagement_Score"], bins=20, kde=True, color="#2b5c8f")
plt.axvline(
    df["Engagement_Score"].mean(),
    color="#d95f02",
    linestyle="--",
    linewidth=1.8,
    label=f"Mean: {df['Engagement_Score'].mean():.2f}",
)
plt.axvline(
    df["Engagement_Score"].median(),
    color="#1b9e77",
    linestyle="-.",
    linewidth=1.8,
    label=f"Median: {df['Engagement_Score'].median():.2f}",
)
plt.axvline(
    50.0,
    color="#e7298a",
    linestyle=":",
    linewidth=2.0,
    label="At Risk Threshold (50.0)",
)
plt.title(
    "Distribution of OTT Session Engagement Score (n = 250)",
    fontsize=13,
    fontweight="bold",
)
plt.xlabel("Engagement Score")
plt.ylabel("Number of Sessions")
plt.legend()
plt.tight_layout()
plt.savefig("../images/01_engagement_distribution.png", dpi=150)
plt.show()

In [ ]:
corrs = (
    df[
        [
            "Watch_Duration_Mins",
            "Monthly_Spend_INR",
            "Subscription_Fee_INR",
            "User_Rating",
            "Engagement_Score",
        ]
    ]
    .corr()["Engagement_Score"]
    .round(3)
)
print("Correlations with Engagement_Score:\n", corrs)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
sns.regplot(
    data=df,
    x="Watch_Duration_Mins",
    y="Engagement_Score",
    ax=axes[0],
    scatter_kws={"alpha": 0.6, "color": "#2b5c8f"},
    line_kws={"color": "#d95f02"},
)
axes[0].set_title(
    f"Watch Duration vs Engagement (r = {corrs['Watch_Duration_Mins']})"
)

sns.regplot(
    data=df,
    x="Monthly_Spend_INR",
    y="Engagement_Score",
    ax=axes[1],
    scatter_kws={"alpha": 0.6, "color": "#1b9e77"},
    line_kws={"color": "#d95f02"},
)
axes[1].set_title(
    f"Monthly Spend vs Engagement (r = {corrs['Monthly_Spend_INR']})"
)

sns.regplot(
    data=df,
    x="User_Rating",
    y="Engagement_Score",
    ax=axes[2],
    scatter_kws={"alpha": 0.6, "color": "#7570b3"},
    line_kws={"color": "#d95f02"},
)
axes[2].set_title(f"User Rating vs Engagement (r = {corrs['User_Rating']})")

plt.tight_layout()
plt.savefig("../images/02_numeric_drivers_vs_engagement.png", dpi=150)
plt.show()

In [ ]:
for col in ["Subscription_Plan", "Device_Type", "Genre", "City"]:
    print(f"\n--- Engagement Score by {col} ---")
    print(
        df.groupby(col)["Engagement_Score"]
        .agg(["count", "mean", "median"])
        .round(2)
    )

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.boxplot(
    data=df,
    x="Subscription_Plan",
    y="Engagement_Score",
    order=["Mobile", "Standard", "Premium"],
    ax=axes[0, 0],
    palette="Blues",
)
axes[0, 0].set_title("Engagement Score by Subscription Plan")

sns.boxplot(
    data=df,
    x="Device_Type",
    y="Engagement_Score",
    order=["Smartphone", "Tablet", "Laptop", "Smart TV"],
    ax=axes[0, 1],
    palette="Greens",
)
axes[0, 1].set_title("Engagement Score by Device Type")

sns.boxplot(
    data=df, x="Genre", y="Engagement_Score", ax=axes[1, 0], palette="Purples"
)
axes[1, 0].set_title("Engagement Score by Genre")

city_order = (
    df.groupby("City")["Engagement_Score"]
    .mean()
    .sort_values(ascending=False)
    .index
)
sns.barplot(
    data=df,
    x="City",
    y="Engagement_Score",
    order=city_order,
    ax=axes[1, 1],
    palette="Oranges_r",
    errorbar=None,
)
axes[1, 1].set_title("Mean Engagement Score by City")
axes[1, 1].set_ylim(0, 80)

plt.tight_layout()
plt.savefig("../images/03_engagement_by_categories.png", dpi=150)
plt.show()

In [ ]:
at_risk = df[df["Engagement_Score"] < 50.0]
print(
    f"At Risk sessions: {len(at_risk)} out of {len(df)}"
    f" ({len(at_risk)/len(df)*100:.1f}%)"
)
print(
    "\nAt Risk breakdown by Subscription Plan:\n",
    at_risk["Subscription_Plan"].value_counts(),
)
print(
    "\nAt Risk breakdown by Device Type:\n",
    at_risk["Device_Type"].value_counts(),
)

plt.figure(figsize=(9, 5))
status_counts = (
    df["Retention_Status"]
    .value_counts()
    .reindex(["At Risk", "Moderate", "High Engagement"])
)
bars = plt.bar(
    status_counts.index,
    status_counts.values,
    color=["#d95f02", "#2b5c8f", "#1b9e77"],
    width=0.55,
)
for bar in bars:
    h = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        h + 2,
        f"{h} ({h/len(df)*100:.1f}%)",
        ha="center",
        fontweight="bold",
    )
plt.title(
    "Session Breakdown by Retention Status (At Risk < 50.0)",
    fontsize=13,
    fontweight="bold",
)
plt.ylabel("Number of Viewing Sessions")
plt.ylim(0, 125)
plt.tight_layout()
plt.savefig("../images/04_retention_status_breakdown.png", dpi=150)
plt.show()

In [ ]:
df.to_csv("../data/processed/ott_movies_data_clean.csv", index=False)
print("Saved cleaned dataset successfully:", df.shape)